# MAVLink Anomaly Detection Using Machine Learning

This notebook implements anomaly detection for MAVLink messages using Isolation Forest and Local Outlier Factor (LOF) algorithms.

In [1]:
# Define the path to the pcap files and the output directory
csv_directory = "../datasets/mavlink"
output_directory = "./model_output"

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (precision_recall_curve)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                           f1_score, roc_auc_score)
import joblib

In [3]:
plt.style.use('ggplot')
sns.set_palette("husl")

## 1. Load Data

In [4]:
def load_mavlink_data(data_dir):
    """Load all CSV files from directory and combine them"""
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"Directory {data_dir} does not exist")

    data_path = Path(data_dir)
    all_data = []
    
    index = 0
    for csv_file in os.listdir(data_dir):
        if csv_file.endswith('.csv'):
            df = pd.read_csv(data_path / csv_file)

            # Print ts column
            print(df['ts'].head())

            # Convert ISO timestamp to Unix timestamp in seconds
            if 'ts' in df.columns:
                # Handle timezone-naive timestamps
                df['ts'] = pd.to_datetime(df['ts'], format='mixed', utc=True)
                # Convert to Unix timestamp (seconds since epoch)
                df['ts'] = df['ts'].astype('int64') // 10**9
            else:
                print(f"Warning: No timestamp column in {csv_file}")
                continue

            # Add source file as a column
            df['source_file'] = csv_file
            all_data.append(df)
            index += 1

            print(f"File Loaded. {index}: {csv_file} - {len(df)} records")
    
    df = pd.concat(all_data, ignore_index=True)
    df.sort_values('ts')
    return df

# Load all data
df = load_mavlink_data(csv_directory)
print(f"Loaded {len(df)} records from {df['source_file'].nunique()} files")

# Display basic statistics
print("\nDataset Overview:")
print(df.info())

/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (82,112) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)
/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (87,98,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)


0    2024-10-31T10:24:00.409577
1    2024-10-31T10:24:00.412284
2    2024-10-31T10:24:00.415254
3    2024-10-31T10:24:00.418361
4    2024-10-31T10:24:00.421173
Name: ts, dtype: object
File Loaded. 1: mavlink_data_20241031_102458.csv - 15126 records
0    2024-11-01T16:01:42.400998
1    2024-11-01T16:01:42.502160
2    2024-11-01T16:01:42.503115
3    2024-11-01T16:01:42.504002
4    2024-11-01T16:01:42.504891
Name: ts, dtype: object
File Loaded. 2: mavlink_data_20241101_160537.csv - 59671 records


/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (87,98,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)
/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (38,88,99,101,120) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)


0    2024-11-01T16:05:46.376728
1    2024-11-01T16:05:46.377817
2    2024-11-01T16:05:46.378830
3    2024-11-01T16:05:46.379839
4    2024-11-01T16:05:46.380489
Name: ts, dtype: object
File Loaded. 3: mavlink_data_20241101_161005.csv - 65792 records
0    2024-10-31T10:29:16.585313
Name: ts, dtype: object
File Loaded. 4: mavlink_data_20241031_102918.csv - 1 records
0    2024-10-31T10:32:30.723126
Name: ts, dtype: object
File Loaded. 5: mavlink_data_20241031_103254.csv - 1 records
0    2024-10-31T10:32:55.433060
1    2024-10-31T10:32:55.440299
2    2024-10-31T10:32:55.446681
3    2024-10-31T10:32:55.449525
4    2024-10-31T10:32:55.451032
Name: ts, dtype: object
File Loaded. 6: mavlink_data_20241031_103524.csv - 2376 records
0    2024-11-01T16:54:42.636975
1    2024-11-01T16:54:42.638160
2    2024-11-01T16:54:42.639609
3    2024-11-01T16:54:42.640617
4    2024-11-01T16:54:42.641963
Name: ts, dtype: object
File Loaded. 7: mavlink_data_20241101_170019.csv - 39119 records


/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (81,92,112) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)
/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (87,98,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)
/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (75) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)


...

0    2024-11-01T16:25:40.468147
1    2024-11-01T16:25:40.470036
2    2024-11-01T16:25:40.471249
3    2024-11-01T16:25:40.472891
4    2024-11-01T16:25:40.474174
Name: ts, dtype: object
File Loaded. 29: mavlink_data_20241101_163432.csv - 135705 records
0    2024-11-01T15:15:20.793985
1    2024-11-01T15:15:20.795816
2    2024-11-01T15:15:20.797392
3    2024-11-01T15:15:20.799646
4    2024-11-01T15:15:20.801689
Name: ts, dtype: object


/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (87,98,118) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)


File Loaded. 30: mavlink_data_20241101_151906.csv - 59743 records


/var/folders/vt/19l5xdgn5dqbwvxb6rrxvyyc0000gn/T/ipykernel_98225/1269410877.py:12: DtypeWarning: Columns (34,82,93,95,114) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / csv_file)


0    2024-11-01T16:41:10.898921
1    2024-11-01T16:41:10.900220
2    2024-11-01T16:41:10.901457
3    2024-11-01T16:41:10.902529
4    2024-11-01T16:41:10.903858
Name: ts, dtype: object
File Loaded. 31: mavlink_data_20241101_164607.csv - 76878 records
Loaded 1251729 records from 31 files

Dataset Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1251729 entries, 0 to 1251728
Columns: 163 entries, MCU_temperature to reason
dtypes: float64(152), int64(1), object(10)
memory usage: 1.5+ GB
None


In [5]:
# Display columns
for col in df.columns:
    print(f"{col}")

MCU_temperature
MCU_voltage
MCU_voltage_max
MCU_voltage_min
Vcc
Vservo
accel_weight
afx
afy
afz
airspeed_variance
alt
alt_ellipsoid
alt_error
altitude
aspd_error
autopilot
base_mode
battery_function
battery_remaining
brkval
charge_state
chunk_seq
clipping_0
clipping_1
clipping_2
cog
compass_variance
coordinate_frame
covariance
current_battery
current_consumed
current_distance
current_height
custom_mode
direction
drop_rate_comm
energy_consumed
eph
epv
error_rp
error_yaw
errors_comm
errors_count1
errors_count2
errors_count3
errors_count4
fault_bitmask
fix_type
flags
freemem
freemem32
h_acc
hdg
hdg_acc
horizontal_fov
id
lat
lat_int
lng
load
loaded
lon
lon_int
mavlink_version
mavpackettype
max_distance
min_distance
mission_mode
mission_state
mode
nav_bearing
nav_pitch
nav_roll
omegaIx
omegaIy
omegaIz
onboard_control_sensors_enabled
onboard_control_sensors_health
onboard_control_sensors_present
orientation
param_count
param_id
param_index
param_type
param_value
pending
pitch
pitchspeed
pos_

In [6]:
# Check for missing values
print("\nMissing values:")
for col in df.columns:
    print(f"{col}: {df[col].isnull().sum()}")


Missing values:
MCU_temperature: 1203671
MCU_voltage: 1203671
MCU_voltage_max: 1203671
MCU_voltage_min: 1203671
Vcc: 1203649
Vservo: 1203649
accel_weight: 1203658
afx: 1229354
afy: 1229354
afz: 1229354
airspeed_variance: 1203677
alt: 1133200
alt_ellipsoid: 1203655
alt_error: 1203651
altitude: 1203603
aspd_error: 1203651
autopilot: 1241964
base_mode: 1241964
battery_function: 1203669
battery_remaining: 1155586
brkval: 1203651
charge_state: 1203669
chunk_seq: 1251527
clipping_0: 1203667
clipping_1: 1203667
clipping_2: 1203667
cog: 1203655
compass_variance: 1203677
coordinate_frame: 1229354
covariance: 1041033
current_battery: 1155586
current_consumed: 1203669
current_distance: 1041033
current_height: 1203666
custom_mode: 1241964
direction: 1204604
drop_rate_comm: 1203646
energy_consumed: 1203669
eph: 1205291
epv: 1205291
error_rp: 1203658
error_yaw: 1203658
errors_comm: 1203646
errors_count1: 1203646
errors_count2: 1203646
errors_count3: 1203646
errors_count4: 1203646
fault_bitmask: 120

## 2. Data Segmentation

In [7]:
def segment_data(df, max_points=100, max_time_diff=0.5):
    """
    Segment data based on:
    - Maximum 100 points per segment
    - Maximum 0.5s time difference within segment
    - Time-ordered continuous data
    
    Args:
        df (pd.DataFrame): Input dataframe with 'ts' column
        max_points (int): Maximum points per segment
        max_time_diff (float): Maximum time difference in seconds
        
    Returns:
        list: List of DataFrame segments
    
    Raises:
        ValueError: If 'ts' column is missing or df is empty
    """
    # Input validation
    if df.empty:
        raise ValueError("Input DataFrame is empty")
    if 'ts' not in df.columns:
        raise ValueError("DataFrame must contain 'ts' column")
    
    # Ensure data is sorted by timestamp
    df = df.sort_values('ts').reset_index(drop=False)
    
    # Find break points in time series
    time_diff = df['ts'].diff()
    time_breaks = np.where(time_diff > max_time_diff)[0]
    
    # Initialize segments
    segments = []
    start_idx = 0
    
    # Create segments based on time breaks and max points
    for break_idx in time_breaks:
        # Handle max points constraint
        current_segment = df.iloc[start_idx:break_idx]
        while len(current_segment) > max_points:
            segments.append(current_segment.iloc[:max_points])
            current_segment = current_segment.iloc[max_points:]
        
        if not current_segment.empty:
            segments.append(current_segment)
        
        start_idx = break_idx
    
    # Handle the final segment
    final_segment = df.iloc[start_idx:]
    while len(final_segment) > max_points:
        segments.append(final_segment.iloc[:max_points])
        final_segment = final_segment.iloc[max_points:]
    
    if not final_segment.empty:
        segments.append(final_segment)
    
    # Validate segments
    for i, segment in enumerate(segments):
        if len(segment) > max_points:
            raise RuntimeError(f"Segment {i} exceeds max_points")
        if (segment['ts'].diff().dropna() > max_time_diff).any():
            raise RuntimeError(f"Time difference violation in segment {i}")
    
    return segments

In [8]:
max_points = 100
max_time_diff = 0.5

segments = segment_data(df, max_points=max_points, max_time_diff=max_time_diff)

In [9]:
print(f"Segmented data into {len(segments)} segments")

Segmented data into 14773 segments


# 3. Segment Labeling 

In [10]:
def label_segments(segments, anomaly_label=-1):
    """Label segments based on presence of anomalies"""
    labeled_segments = []
    for segment in segments:
        # Standard label is 0 (normal)
        segment['segment_label'] = 0
        # If any point is anomalous, mark as 1 (anomaly)
        if (segment['anomaly_label'] == anomaly_label).any():
            segment['segment_label'] = 1
        labeled_segments.append(segment)
    return labeled_segments

In [11]:
labeled_segments = label_segments(segments)

In [12]:
print(f"Labeled {len(labeled_segments)} segments")
print(f"Anomalous segments: {sum([s['segment_label'].iloc[0] == -1 for s in labeled_segments])}")

Labeled 14773 segments
Anomalous segments: 0


# 4. Train/Test Split for Segments

In [13]:
def split_segments(labeled_segments):
    """
    Split segments into train/test/validate:
    - Normal: 60/20/20
    - Anomaly: 50/50 (test/validate only)
    """
    normal_segments = [s for s in labeled_segments if s['segment_label'].iloc[0] == 1]
    anomaly_segments = [s for s in labeled_segments if s['segment_label'].iloc[0] == -1]
    
    # Split normal segments
    n_train = int(len(normal_segments) * 0.6)
    n_test = int(len(normal_segments) * 0.2)
    
    train_segments = normal_segments[:n_train]
    test_segments = normal_segments[n_train:n_train+n_test]
    val_segments = normal_segments[n_train+n_test:]
    
    # Split anomaly segments
    n_test_anomaly = len(anomaly_segments) // 2
    test_segments.extend(anomaly_segments[:n_test_anomaly])
    val_segments.extend(anomaly_segments[n_test_anomaly:])
    
    return train_segments, test_segments, val_segments

In [14]:
train_segments, test_segments, val_segments = split_segments(labeled_segments)

# 5. Feature Engineering

In [15]:
def engineer_features(df):
    """
    Feature engineering without synthetic data
    
    Features generated:
    - gps_speed_3d: 3D speed calculated from vx, vy, vz
    - heading_stable: Heading stability over time window
    - height_consistency: Consistency check between different altitude measures
    - position_precision: GPS position precision metrics
    """
    features = pd.DataFrame()
    
    # GPS speed and movement features
    gps_cols = ['lat', 'lon', 'alt', 'vx', 'vy', 'vz']
    if all(col in df.columns for col in gps_cols):
        # 3D speed
        features['gps_speed_3d'] = np.sqrt(
            df['vx'].fillna(0)**2 + 
            df['vy'].fillna(0)**2 + 
            df['vz'].fillna(0)**2
        )
        
        # Heading stability (requires timestamp)
        if 'ts' in df.columns:
            features['heading_stability'] = df.groupby(df['ts'].diff().gt(0.5).cumsum())['hdg'].std().fillna(0)
        
        # Height consistency
        if 'relative_alt' in df.columns:
            features['height_consistency'] = np.abs(df['alt'] - df['relative_alt']).fillna(0)
        
        # Position precision
        if 'fix_type' in df.columns:
            features['position_precision'] = df['fix_type'].fillna(0)
    
    # Handle missing values
    features = features.fillna(0)
    features = features.replace([np.inf, -np.inf], 0)
    
    return features

In [16]:
# Feature engineering
X_train = pd.concat([engineer_features(s) for s in train_segments])
X_test = pd.concat([engineer_features(s) for s in test_segments])
X_val = pd.concat([engineer_features(s) for s in val_segments])

y_test = pd.concat([s['segment_label'] for s in test_segments])
y_val = pd.concat([s['segment_label'] for s in val_segments])

# 6. Model Training

In [17]:
from sklearn.model_selection import KFold

def standardize_labels(labels):
    """
    Convert labels to binary 0/1 format.
    For anomaly detection, typically:
    - Normal: 1 or 1.0 -> 0
    - Anomaly: -1 or 0.0 -> 1
    """
    labels = np.asarray(labels)
    # Convert -1/1 to 0/1 if needed
    if -1 in labels:
        labels = (labels == -1).astype(int)
    return labels

def find_best_threshold(y_true, scores, cv_folds=None):
    """
    Find the optimal threshold that maximizes the F1 score.
    
    Args:
        y_true (array-like): True binary labels
        scores (array-like): Predicted scores/probabilities
        cv_folds (list): Optional list of (train_idx, val_idx) for cross-validation
        
    Returns:
        float: Best threshold value
        float: Best F1 score achieved
        dict: Additional metrics (precision, recall at best threshold)
    
    Note:
        Uses precision-recall curve for efficient threshold search
    """
    # Input validation
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    
    if len(y_true) != len(scores):
        raise ValueError("Length of y_true and scores must match")
    if len(y_true) == 0:
        raise ValueError("Input arrays cannot be empty")
        
    if cv_folds is None:
        # Use all data if no CV provided
        cv_folds = [(np.arange(len(y_true)), np.arange(len(y_true)))]
        
    # Find best threshold using cross-validation
    best_thresholds = []
    best_f1s = []
    
    for train_idx, val_idx in cv_folds:
        # Get precision-recall curve
        precision, recall, thresholds = precision_recall_curve(
            y_true[val_idx], 
            scores[val_idx]
        )
        
        # Calculate F1 for each threshold
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
        
        # Find best threshold
        best_idx = np.argmax(f1_scores)
        if len(thresholds) > best_idx:
            best_thresholds.append(thresholds[best_idx])
            best_f1s.append(f1_scores[best_idx])
        else:
            # Handle edge case where best point is at recall=1
            best_thresholds.append(thresholds[-1])
            best_f1s.append(f1_scores[-1])
    
    # Average thresholds across folds
    best_threshold = np.mean(best_thresholds)
    best_f1 = np.mean(best_f1s)
    
    # Calculate final metrics
    final_preds = (scores > best_threshold).astype(int)
    final_f1 = f1_score(y_true, final_preds)
    
    metrics = {
        'f1_score': final_f1,
        'f1_cv_mean': best_f1,
        'f1_cv_std': np.std(best_f1s),
        'threshold_cv_std': np.std(best_thresholds)
    }
    
    return best_threshold, final_f1, metrics

def find_optimal_threshold(model, X_val, y_val, n_splits=5):
    """
    Find optimal threshold for anomaly detection model using cross-validation.
    
    Args:
        model: Fitted anomaly detection model
        X_val: Validation features
        y_val: True validation labels
        n_splits: Number of cross-validation splits
        
    Returns:
        float: Optimal threshold
        dict: Performance metrics
    """
    # Get model scores
    if hasattr(model, 'score_samples'):
        scores = model.score_samples(X_val)
    else:
        scores = model.decision_function(X_val)
    
    # Prepare cross-validation
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_folds = list(kf.split(X_val))
    
    # Find best threshold
    threshold, f1, metrics = find_best_threshold(
        y_val, 
        scores, 
        cv_folds=cv_folds
    )
    
    print(f"Best threshold: {threshold:.3f}")
    print(f"F1 Score: {f1:.3f}")
    print("\nCross-validation metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.3f}")
        
    return threshold, metrics

def evaluate(y_true, y_pred, model_name="Model"):
    """
    Evaluate and print model performance metrics.
    """
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1v = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_pred)
    except:
        auc = float('nan')
    print(f"{model_name} - Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1v:.4f}, AUC: {auc:.4f}")
    return prec, rec, f1v, auc

## 1. Isolation Forest

In [18]:
iso = IsolationForest(
    n_estimators=300,
    max_samples='auto',
    contamination=0.1,
    random_state=42
)

iso.fit(X_train)

# Find the best threshold based on validation set
best_iso_threshold, metrics = find_optimal_threshold(
    model=iso,
    X_val=X_val,
    y_val=y_val, n_splits=5)
print(f"Best Isolation Forest threshold: {best_iso_threshold:.4f}")

# Predict on test set
iso_test_scores = -iso.score_samples(X_test)
iso_preds_test = (iso_test_scores > best_iso_threshold).astype(int)

# Evaluate Isolation Forest
evaluate(y_test, iso_preds_test, "Isolation Forest")

# Accuracy
iso_accuracy = accuracy_score(y_test, iso_preds_test)
print(f"Isolation Forest - Accuracy: {iso_accuracy:.4f}")

Best threshold: -0.823
F1 Score: 1.000

Cross-validation metrics:
f1_score: 1.000
f1_cv_mean: 1.000
f1_cv_std: 0.000
threshold_cv_std: 0.000
Best Isolation Forest threshold: -0.8232
Isolation Forest - Precision: 1.0000, Recall: 1.0000, F1: 1.0000, AUC: nan
Isolation Forest - Accuracy: 1.0000


## 2. Local Outlier Factor

In [19]:
lof = LocalOutlierFactor(
    n_neighbors=35,
    algorithm='ball_tree',
    leaf_size=30,
    metric='minkowski',
    contamination=0.05,
    novelty=True
)

lof.fit(X_train)

# Find the best threshold based on validation set
best_lof_threshold, metrics = find_optimal_threshold(
    model=lof,
    X_val=X_val,
    y_val=y_val, n_splits=5)
print(f"Best LOF threshold: {best_lof_threshold:.4f}")

# Predict on test set
lof_test_scores = -lof.decision_function(X_test)
lof_preds_test = (lof_test_scores > best_lof_threshold).astype(int)

# Evaluate LOF
evaluate(y_test, lof_preds_test, "Local Outlier Factor")

# Accuracy
lof_accuracy = accuracy_score(y_test, lof_preds_test)


/Users/dogrod/Developer/UVIC/meng-project/.venv/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


Best threshold: -10381.744
F1 Score: 1.000

Cross-validation metrics:
f1_score: 1.000
f1_cv_mean: 1.000
f1_cv_std: 0.000
threshold_cv_std: 8.134
Best LOF threshold: -10381.7437


/Users/dogrod/Developer/UVIC/meng-project/.venv/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


Local Outlier Factor - Precision: 1.0000, Recall: 1.0000, F1: 1.0000, AUC: nan


# 7. Export model

In [20]:
# Save model and preprocessing info
model_artifacts = {
    'model': iso,
    'threshold': best_iso_threshold,
    'max_points': max_points,
    'max_time_diff': max_time_diff,
}
joblib.dump(model_artifacts, 'mavlink_model_artifacts.pkl')

['mavlink_model_artifacts.pkl']